# Les erreurs de forme

| Module | Durée | Prérequis |
|---|---|---|
| M0 – Python et NumPy comme outils | 75 min | aucun |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Une cantine scolaire prépare trois recettes. Une collègue a écrit un petit programme pour calculer **le coût d'ingrédients d'une portion de chaque recette**. Son programme contient au moins **trois erreurs**. Certaines déclenchent un message d'erreur ; d'autres, plus dangereuses, donnent un résultat faux sans rien signaler.

Votre mission : les trouver, comprendre leur cause et les corriger.

Les données :

- les **quantités** par portion (en grammes), pour 3 recettes et 4 ingrédients : farine, œufs, lait, sucre ;
- les **prix** des ingrédients (en euros par kilogramme).

## ② Estimer avant de calculer

Une portion de crêpes, de gâteau ou de flan coûte, en ingrédients, quelques dizaines de centimes. **Gardez cet ordre de grandeur en tête** : c'est votre meilleur détecteur d'erreurs.

In [ ]:
pari_cout_max = 1.0    # coût maximal plausible d'une portion, en euros (modifiez)

## ③ Abstraire : une méthode de débogage

Quand un calcul échoue ou donne un résultat suspect, vérifiez dans cet ordre :

1. **Les formes** : afficher `.shape` de chaque tableau. Les dimensions sont-elles compatibles ?
2. **Les unités** : grammes ou kilogrammes ? euros ou centimes ?
3. **La forme du résultat** : on attend un coût par recette, donc 3 nombres. En a-t-on 3 ?
4. **Un petit cas calculé à la main** : une recette, deux ingrédients.
5. **Isoler** : exécuter le calcul morceau par morceau pour trouver l'étape fautive.

## ④ Calculer : le programme de la collègue

In [ ]:
import numpy as np

recettes = ["crêpes", "gâteau", "flan"]
ingredients = ["farine", "œufs", "lait", "sucre"]

# Quantités par portion, en grammes : une ligne par recette, une colonne par ingrédient
Q = np.array([
    [40, 30, 100, 10],   # crêpes
    [50, 45,   0, 50],   # gâteau
    [10, 55, 120, 25],   # flan
])

# Prix en euros par kilogramme : farine, œufs, lait (la collègue a oublié le sucre)
prix = np.array([1.2, 6.5, 1.0])

### Erreur n° 1

Le coût d'une recette, c'est la somme des produits « quantité × prix » sur ses ingrédients :

$$
\text{coût} = \sum_{\text{ingrédients}} \text{quantité} \times \text{prix}
$$

La collègue utilise le **produit matriciel** `@`, qui fait exactement cela pour chaque ligne de `Q`.

![Schéma : Q de forme (3, 4) multiplié par prix de forme (4,) donne un vecteur de forme (3,)](../../assets/schemas/schema-produit-matrice-vecteur.svg)

Exécutez la cellule.

In [ ]:
cout = Q @ prix
print(cout)

**Activité** : Lisez le message d'erreur : il cite des dimensions. Affichez `Q.shape` et `prix.shape`, puis expliquez en une phrase la cause de l'erreur.

In [ ]:
print(Q.shape, prix.shape)

Indiquez vos réponses et justifiez-les.

**Activité** : Corrigez le vecteur des prix : le sucre coûte 1,1 € par kilogramme.

In [ ]:
prix = np.array([1.2, 6.5, 1.0, 1.1])   # corrigé
cout = Q @ prix
print(dict(zip(recettes, cout.round(2).tolist())))

### Erreur n° 2

Plus d'erreur affichée. Mais regardez les valeurs : plusieurs centaines d'euros pour une portion de crêpes ? Comparez à votre pari.

**Activité** : D'où vient ce facteur environ 1 000 ? Utilisez l'étape 2 de la méthode.

Indiquez vos réponses et justifiez-les.

In [ ]:
# Correction : les quantités sont en grammes, les prix en euros par KILOgramme
cout = (Q / 1000) @ prix
print(dict(zip(recettes, cout.round(3).tolist())))

### Erreur n° 3

Pour comprendre d'où vient le coût, la collègue a écrit une seconde version, « sans produit matriciel ». Elle multiplie les tableaux élément par élément, puis additionne.

In [ ]:
detail = (Q / 1000) * prix     # coût de chaque ingrédient dans chaque recette
cout_v2 = detail.sum(axis=0)
print(cout_v2)

**Activité** : Aucune erreur n'est signalée. Pourtant le résultat est faux. Appliquez l'étape 3 de la méthode : combien de nombres attend-on ? combien en obtient-on ? Que représentent-ils ?

Indiquez vos réponses et justifiez-les.

In [ ]:
cout_v2 = detail.sum(axis=1)   # corrigé : on additionne le long de chaque ligne (chaque recette)
print(dict(zip(recettes, cout_v2.round(3).tolist())))
print("Les deux versions concordent :", np.allclose(cout, cout_v2))

**Pourquoi la ligne `detail = (Q / 1000) * prix` fonctionne-t-elle**, alors que `Q` a pour forme (3, 4) et `prix` (4,) ? NumPy « étire » automatiquement le vecteur des prix sur chaque ligne : c'est la **diffusion** (*broadcasting*). C'est pratique, mais c'est aussi pour cela que certaines erreurs passent inaperçues.

## ⑤ Interpréter

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5))
bas = np.zeros(len(recettes))
for j, ing in enumerate(ingredients):
    ax.bar(recettes, detail[:, j], bottom=bas, label=ing)
    bas += detail[:, j]
ax.set_ylabel("coût par portion (€)")
ax.legend()
plt.show()
print("Votre pari de coût maximal :", pari_cout_max, "€ ; coût maximal calculé :", cout.max().round(3), "€")

**Activité** : Quelle recette est la plus chère par portion ? Quel ingrédient pèse le plus dans son coût ? Ce résultat vous surprend-il ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Classer les erreurs.** Parmi les trois erreurs, lesquelles auraient été détectées sans vérification humaine ? Laquelle est la plus dangereuse, et pourquoi ?
- **Prévenir.** Ajoutez une vérification automatique qui arrête le calcul si les formes ne sont pas compatibles (indice : `assert`).
- **Généraliser.** Écrivez une fonction `cout_par_portion(Q_grammes, prix_kg)` qui renvoie le coût de chaque recette, avec cette vérification.

In [ ]:
def cout_par_portion(Q_grammes, prix_kg):
    # À compléter : vérifier les formes, convertir les unités, calculer
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def cout_par_portion(Q_grammes, prix_kg):
    Q_grammes, prix_kg = np.asarray(Q_grammes, dtype=float), np.asarray(prix_kg, dtype=float)
    assert Q_grammes.ndim == 2 and prix_kg.ndim == 1, "Q doit être un tableau 2D et les prix un vecteur"
    assert Q_grammes.shape[1] == prix_kg.shape[0], (
        f"{Q_grammes.shape[1]} ingrédients dans Q mais {prix_kg.shape[0]} prix")
    return (Q_grammes / 1000) @ prix_kg

print(cout_par_portion(Q, prix).round(3))
try:
    cout_par_portion(Q, [1.2, 6.5, 1.0])
except AssertionError as err:
    print("Erreur détectée :", err)

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Isoler la cause d'un problème de fonctionnement": 1,
    "Résoudre un problème de fonctionnement": 1,
    "Sentir si le résultat est plausible pour le contexte": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "0.3-erreurs-de-forme", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")